In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from feature_engine.encoding import CountFrequencyEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import shap
import mlflow.sklearn
import mlflow

import config

In [ ]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw
#mlflow.enable_system_metrics_logging()

In [ ]:
mlflow.set_experiment("T0083 - Audit Review")

with mlflow.start_run(run_name=f"Sklearn-RF Model", nested=True) as greatgrandparent_run:
    for customer in config.customerList:
        for auditLog in config.auditList:
            with mlflow.start_run(run_name=f"Base context-{auditLog}-{customer}", nested=True) as grandparent_run:
                filepath = Path(f'{config.data_location}FS_base_data/{auditLog}/FS_base_data_{auditLog}_{customer}.parquet')
                if filepath.is_file()==True:
                    df = pd.read_parquet(filepath, engine='pyarrow')
                    #df = df.drop(['T0083','T0083_SEVERITY','T_MLAR','T_MLAR_SEVERITY','MLAR_TYPE'],axis=1) #RK should be already done in FS scripts
                    #df = df.astype(str).replace('nan',np.nan).dropna(how='all',axis=1) #RK should be already done in FS scripts
                    df = df.astype(str)
                    features = df.columns.tolist()
                    #Update to Anomaly field onced changed
                    categoricalColumns = df.drop(['T0083_SEVERITY'],axis=1).select_dtypes(include=['object']).columns.tolist()
                    encoder = CountFrequencyEncoder(encoding_method='frequency', variables=categoricalColumns, ignore_format=True)
                    dataFrame = encoder.fit_transform(df) 
                    #Update to Anomaly field onced changed
                    X_train, X_test, y_train, y_test = train_test_split(
                        dataFrame[dataFrame.columns.drop(list(dataFrame.filter(like='T0083_SEVERITY')))], 
                        dataFrame.filter(like='T0083_SEVERITY').sum(axis=1), 
                        test_size=0.30, 
                        random_state=1234)
                            
                    # Define model parameters
                    # classWeight = "balanced"
                    maxDepth = 5

                    # Initialize the model
                    rf_model = RandomForestClassifier(max_depth=maxDepth)
                    
                    # Start an MLflow run
                    with mlflow.start_run(run_name=f"Model Evaluation", nested=True) as parent_run:
                        # Log model parameters and train
                        #mlflow.log_param("class_weight",classWeight)
                        #mlflow.log_param("max_depth",maxDepth)
                        rf_model.fit(X_train.values,y_train.values) 
                        
                        # Predict anomalies and generate histogram
                        predicted = pd.DataFrame(rf_model.predict(X_test.values)) 
                        
                        # Combine the scores with the encoded data
                        predictedDF = pd.concat([pd.DataFrame(data=predicted,columns=['anomaly']),dataFrame],axis=1)
                        predictedDF = predictedDF[predictedDF['anomaly'] > 0]
                        #predictedDF = predictedDF.iloc[:100,1:]

                        #Export Dataset
                        mlflow.log_table(predictedDF,"predictedDF.json")

                        with mlflow.start_run(run_name=f"Model Explaination", nested=True) as child_run:
                            #Log metrics
                            precisionScore = metrics.precision_score(y_test, predicted, average='micro',zero_division=1)
                            recallScore = metrics.recall_score(y_test, predicted, average='micro',zero_division=1)
                            f1Score = metrics.f1_score(y_test, predicted, average='micro',zero_division=1)               
                            classificationReport = metrics.classification_report(y_test, predicted,zero_division=1)
                            mlflow.log_metric("precision", precisionScore)
                            mlflow.log_metric("recall", recallScore)
                            mlflow.log_metric("f1_score", f1Score)
                            mlflow.log_text(classificationReport, "classificationReport.txt")
                            
                            #Train explaination model
                            #custom_masker = shap.maskers.Partition(dataFrame, max_samples=30, clustering='correlation')
                            #explainer = shap.Explainer(rf_model.predict,masker=custom_masker) #, dataFrame) max_eval=500,nsamples=10
                            explainer = shap.TreeExplainer(rf_model,feature_names=features) #,feature_names=features 
                            shap_values = explainer(dataFrame)
                            feature_names = shap_values.feature_names
                            
                            ## plot generation
                            shapBarplot = plt.figure(figsize=(9, 3))
                            shap.summary_plot(shap_values,features=explainer.feature_names,max_display=12, plot_type="bar")
                            mlflow.log_figure(shapBarplot, f"{customer}_GlobalBarPlot.png")  
                            logModel = mlflow.sklearn.log_model(sk_model=rf_model,artifact_path=f"Sklearn-RF_{auditLog}_{customer}_base_model",registered_model_name=f"Sklearn-RF_{auditLog}_{customer}_base_model")
                else:
                    print("Error - Feature store file not found. Check directory.")

In [ ]:
import mlflow
mlflow.end_run()